In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_groq import ChatGroq

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b")

Writing function to retrive chat history for a specific session_id

In [7]:
store={}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(llm,get_session_history)

Specify the configuration.

In [5]:
config1={"session_id":"chat1"}

Invoke the above function.

In [8]:
from langchain_core.messages import HumanMessage
response1=with_message_history.invoke(
[HumanMessage(content="Hi my name is Atul Mangla. I am a AI Engineer")],config=config1)

In [10]:
response1.content

'Hello Atul! 👋 Great to meet an AI Engineer. How can I assist you today? Whether you have questions about models, need help with a project, or just want to chat about the latest in AI, I’m here to help.'

Check the store for finding the chat history using session

In [11]:
print(store["chat1"].messages)

[HumanMessage(content='Hi my name is Atul Mangla. I am a AI Engineer', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Atul! 👋 Great to meet an AI Engineer. How can I assist you today? Whether you have questions about models, need help with a project, or just want to chat about the latest in AI, I’m here to help.', additional_kwargs={'reasoning_content': 'The user says "Hi my name is Atul Mangla. I am a AI Engineer". Likely they just introduce themselves. The assistant should respond politely, perhaps ask how can help. No disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 85, 'total_tokens': 192, 'completion_time': 0.228421267, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.015955874, 'prompt_tokens_details': None, 'queue_time': 0.067287348, 'total_time': 0.244377141}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand

Check ur name again using the same config by invoking the above function.

In [12]:
response2=with_message_history.invoke(
[HumanMessage(content="What is my name")],
config=config1
)
response2.content

'Your name is **Atul\u202fMangla**.'

In [18]:
print(store["chat1"])

Human: Hi my name is Atul Mangla. I am a AI Engineer
AI: Hello Atul! 👋 Great to meet an AI Engineer. How can I assist you today? Whether you have questions about models, need help with a project, or just want to chat about the latest in AI, I’m here to help.
Human: What is my name
AI: Your name is **Atul Mangla**.


## AI Assistant Loop

In [21]:
# Reusable assistant loop using your existing store/get_session_history and with_message_history
from langchain_core.messages import HumanMessage, SystemMessage

def assistant_loop(session_id: str, system_prompt: str = None):
    # ensure history exists
    history = get_session_history(session_id)

    if system_prompt:
        # record system instruction (only once)
        with_message_history.invoke([SystemMessage(content=system_prompt)], config={"session_id": session_id})

    print(f"Assistant session '{session_id}' started. Type 'exit' to quit.")
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit"):
            print("Exiting session.")
            break

        # call the runnable which will automatically read/update history
        resp = with_message_history.invoke([HumanMessage(content=user_input)], config={"session_id": session_id})
        print("Assistant:", resp.content)
        print("-" * 40)

# Example usage:
# assistant_loop("chat1", system_prompt="You are a helpful assistant.")

In [22]:
assistant_loop("chat1", system_prompt="You are a helpful assistant.")

Assistant session 'chat1' started. Type 'exit' to quit.
Assistant: Hello! You’re **Atul Mangla**, an AI Engineer. How can I assist you today?
----------------------------------------
Assistant: Glad to hear it! 😊 Let me know if there’s anything you’d like to discuss—whether it’s a LangChain project, a new model you’re exploring, or anything else on your mind. I'm here to help!
----------------------------------------
Exiting session.
